In [1]:
# notebook to explore variants/assocations in GTEx

In [1]:
import pandas as pd
import os
from tqdm import tqdm

In [2]:
# open data frame from Tian for re-org/pulling lead variant info
tian_raw = pd.read_csv(
    'gtex_meta_result_with_global_eqtl_stats.tsv',
    sep = '\t'
)
# reformat id to match GTEx
tian_raw.loc[:, 'gtex_id'] = [('_').join(i.split(':')) + '_b38' for i in tian_raw['Variant_ID']]

In [3]:
# filter for lead variants
leadVars = tian_raw[tian_raw['Variant_Tag'] == 'lead'].copy()
# print the length of all variants
print(f'there are {len(tian_raw['Variant_ID'].unique())} unique variants in the full df')
# print the number of lead variants
print(f'there are {len(leadVars['Leadvar'].unique())} unique lead variants in the full df')

there are 4484 unique variants in the full df
there are 631 unique lead variants in the full df


In [4]:
# make a column with gtex_id_Tissue_name_gene as an id for filtering/parsing
leadVars.loc[:, 'filter_id'] = [(':').join([var, tissue, gene]) for var, tissue, gene in zip(leadVars['gtex_id'], leadVars['Tissue_Name'], leadVars['Gene_Name'])]
# subset for only single instances
uniqueQTLs = leadVars.filter(['gtex_id', 'Tissue_Name', 'Gene_Name', 'filter_id']).drop_duplicates()
# 625 unique variants with 1,390 unique variant - gene - tissue pairs

In [5]:
# store the path to the gtex eqtl data as a variable
GTEx = '/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/promoter_sat_mut_comp/raw_data/gtex_v10_fineMapping/GTEx_Analysis_v10_eQTL_updated'
# store the path to the finemapping as a variable
fineMap = '/projects/tewhey-lab/buttsj/Variant_Effects/revision_experiments/promoter_sat_mut_comp/raw_data/gtex_v10_fineMapping/SuSiE_fineMapped'

In [6]:
tian_raw.head()

,Variant_ID,Leadvar,Variant_Tag,Subject_ID,Tissue_Abbr,Tissue_Name,Genotype(ref0/alt1),TPM,skew_pred,cell_type,enhancer_ids,Gene_Name,eqtl_beta,eqtl_p,gtex_id
0,chr1:906824:C:T,chr1:906982:C:T,other,GTEX-111CU,ESOMCS,Esophagus_Mucosa,0,0.1153,0.254888,sknsh,EH38E2776603,ENSG00000230699,-0.054612,0.001405,chr1_906824_C_T_b38
1,chr1:906824:C:T,chr1:906982:C:T,other,GTEX-111VG,CLFIB,Cells_Cultured_fibroblasts,0,1.6757,0.254888,sknsh,EH38E2776603,ENSG00000230699,-0.595532,0.003868,chr1_906824_C_T_b38
2,chr1:906824:C:T,chr1:906982:C:T,other,GTEX-111YS,CLFIB,Cells_Cultured_fibroblasts,0,1.6548,0.254888,sknsh,EH38E2776603,ENSG00000230699,-0.595532,0.003868,chr1_906824_C_T_b38
3,chr1:906824:C:T,chr1:906982:C:T,other,GTEX-111YS,ESOMCS,Esophagus_Mucosa,0,0.1194,0.254888,sknsh,EH38E2776603,ENSG00000230699,-0.054612,0.001405,chr1_906824_C_T_b38
4,chr1:906824:C:T,chr1:906982:C:T,other,GTEX-1122O,CLFIB,Cells_Cultured_fibroblasts,0,2.2505,0.254888,sknsh,EH38E2776603,ENSG00000230699,-0.595532,0.003868,chr1_906824_C_T_b38


In [7]:
# iterate through each tissue and pull those variants raw data
tissues2cat = []
for tissue in tqdm(uniqueQTLs['Tissue_Name'].unique()):
    # filter for only variants from that tissue
    tissueVars = uniqueQTLs[uniqueQTLs['Tissue_Name'] == tissue].copy()
    # load eGene pairs
    eGenePairs = pd.read_csv(f'{GTEx}/{tissue}.v10.eGenes.txt.gz', sep = '\t')
    # make a dictionary for matching gene_id to gene_name
    geneDict = dict(zip(
        eGenePairs['gene_id'],
        eGenePairs['gene_name']
    ))
    # load the parquets of significant pairs
    gtexSignifPairs = pd.read_parquet(f'{GTEx}/{tissue}.v10.eQTLs.signif_pairs.parquet')
    # add gene name
    gtexSignifPairs.loc[:, 'gene_name'] = [geneDict.get(i) for i in gtexSignifPairs['gene_id']]
    # load the fine mapping data
    fineMapping = pd.read_parquet(f'{fineMap}/{tissue}.v10.eQTLs.SuSiE_summary.parquet')
    print(f'loaded all gtex data: {tissue}')
    # make lists for storing dfs before concantenating
    dfs2Cat = []
    # iterate through all variants for that tissue
    for var in tqdm(tissueVars['filter_id']):
        # get variant id
        varID = var.split(':')[0]
        # get gene name
        geneName = var.split(':')[-1]
        # filter fineMapping for that variant and gene
        varFineMap = fineMapping[(fineMapping['gene_name'] == geneName) & (fineMapping['variant_id'] == varID)].copy()
        # add tissue to the df for keeping track
        varFineMap.loc[:, 'tissue'] = [tissue for i in range(len(varFineMap))]
        # filter the gtex signifPairs for that variant and gene
        varGTExSig = gtexSignifPairs[(gtexSignifPairs['variant_id'] == varID) & (gtexSignifPairs['gene_name'] == geneName)].copy()
        if len(varGTExSig) == 1:
            # merge data with finemapping
            merged = varFineMap.merge(varGTExSig.filter(['variant_id', 'tss_distance', 'rna_samples', 'rna_count',
                                                         'pval_nominal', 'slope', 'slope_se', 'pval_nominal_threshold',
                                                         'min_pval_nominal', 'pval_beta']), how='inner', on='variant_id')
        elif len(varGTExSig) == 0:
            print(f'{varID} is not a significant hit in {tissue}')
            # make a dummy df to merge
            dummyDF = pd.DataFrame({
                'variant_id' : [varID],
                'tss_distance' : ['not_sig'],
                'rna_samples' : ['not_sig'],
                'rna_count' : ['not_sig'],
                'pval_nominal' : ['not_sig'],
                'slope' : ['not_sig'],
                'slope_se' : ['not_sig'],
                'pval_nominal_threshold' : ['not_sig'],
                'min_pval_nominal' : ['not_sig'],
                'pval_beta' : ['not_sig']
            })
            # merge dummy df with fine mapping
            merged = varFineMap.merge(dummyDF, how='inner', on='variant_id')
        else:
            print('you got a problem')
        # append merged to dataframe
        dfs2Cat.append(merged)
    # concatenate data for that tissue and add to list
    tissues2cat.append(pd.concat(dfs2Cat))
# concatenate all together
allVarGTExPlusFineMap = pd.concat(tissues2cat).reset_index(drop=True)

  0%|          | 0/40 [00:00<?, ?it/s]

loaded all gtex data: Esophagus_Mucosa


  2%|▎         | 1/40 [00:29<18:52, 29.03s/it]

loaded all gtex data: Cells_Cultured_fibroblasts


chr15_25884652_C_T_b38 is not a significant hit in Cells_Cultured_fibroblasts


...

 95%|█████████▌| 38/40 [10:01<00:20, 10.36s/it]

loaded all gtex data: Minor_Salivary_Gland


 98%|█████████▊| 39/40 [10:03<00:07,  7.92s/it]

loaded all gtex data: Uterus


100%|██████████| 40/40 [10:05<00:00, 15.13s/it]


In [8]:
# save that df to disk
allVarGTExPlusFineMap.to_csv('all_highPIP_with_GTEx_Tissue_Data.tsv', sep = '\t', index = False)

In [4]:
# show the header of the fine mapping file
!parquet-tools show --head 1 {fineMap}/Spleen.v10.eQTLs.SuSiE_summary.parquet


+-------------------+-------------+------------------------+---------------------+----------+----------+---------+-----------+-------+----------+
| phenotype_id      | gene_name   | biotype                | variant_id          |      pip |       af |   cs_id |   cs_size |   afc |   afc_se |
|-------------------+-------------+------------------------+---------------------+----------+----------+---------+-----------+-------+----------|
| ENSG00000227232.5 | WASH7P      | unprocessed_pseudogene | chr1_665098_G_A_b38 | 0.197638 | 0.128159 |       1 |         2 |   nan |      nan |
+-------------------+-------------+------------------------+---------------------+----------+----------+---------+-----------+-------+----------+


In [11]:
# show the header of the GTEx file
!parquet-tools show --head 1 {GTEx}/Adipose_Subcutaneous.v10.eQTLs.signif_pairs.parquet

+-------------------+--------------------+----------------+-----------+--------------+------------+----------------+----------+------------+--------------------------+--------------------+-------------+
| gene_id           | variant_id         |   tss_distance |        af |   ma_samples |   ma_count |   pval_nominal |    slope |   slope_se |   pval_nominal_threshold |   min_pval_nominal |   pval_beta |
|-------------------+--------------------+----------------+-----------+--------------+------------+----------------+----------+------------+--------------------------+--------------------+-------------|
| ENSG00000227232.5 | chr1_64764_C_T_b38 |          35211 | 0.0675105 |           93 |         96 |    3.05258e-10 | 0.554569 |  0.0866886 |              0.000606123 |        3.05258e-10 | 5.74521e-07 |
+-------------------+--------------------+----------------+-----------+--------------+------------+----------------+----------+------------+--------------------------+--------------------+

In [7]:
# check on chr15_99169776_G_A_b38
# pull the ID from the finemapping for Spleen
!parquet-tools show {fineMap}/Spleen.v10.eQTLs.SuSiE_summary.parquet | grep 'chr15_99169776_G_A_b38'
# pull the ID from teh eQTL data for Spleen
!parquet-tools show {GTEx}/Spleen.v10.eQTLs.signif_pairs.parquet | grep 'chr15_99169776_G_A_b38'

| ENSG00000182253.15 | SYNM            | protein_coding                     | chr15_99169776_G_A_b38                                                                                                                                                | 0.973185    | 0.427798   |       2 |         1 |  -0.466309    |   0.120706  |


In [8]:
# check on chr15_99169776_G_A_b38
# pull the ID from the finemapping for Spleen
!parquet-tools show {fineMap}/Adipose_Subcutaneous.v10.eQTLs.SuSiE_summary.parquet | grep 'chr15_99169776_G_A_b38'
# pull the ID from teh eQTL data for Spleen
!parquet-tools show {GTEx}/Adipose_Subcutaneous.v10.eQTLs.signif_pairs.parquet | grep 'chr15_99169776_G_A_b38'

| ENSG00000182253.15 | chr15_99169776_G_A_b38                                                                                                                                                                                       |          71559 | 0.479606   |          524 |        682 |   2.95162e-21  | -0.271238  | 0.027652   |              0.000111198 |       5.09332e-61  | 1.71827e-53  |
| ENSG00000261054.1  | chr15_99169776_G_A_b38                                                                                                                                                                                       |          40410 | 0.479606   |          524 |        682 |   3.24939e-07  | -0.191624  | 0.0371159  |              0.000100923 |       5.008e-22    | 1.85685e-17  |
| ENSG00000261616.1  | chr15_99169776_G_A_b38                                                                                                                                                                       